# Big Mart Sales Prediction - Regression
**Team Maverick** | Predicting the sales of a product in a store

**Flow of this notebook:** Dataset -> Preprocessing -> Feature Engineering -> EDA -> Multicollinearity (VIF) -> Encoding -> Model Training -> Predictions -> Our Final (Structural) Model

## Dataset Information
BigMart collected 2013 sales data for 1559 products across 10 stores in different cities. Each row is **one product in one store**. The aim is to predict `Item_Outlet_Sales` and understand which product and store properties drive sales.

Variable | Description
----------|--------------
Item_Identifier | Unique product ID (FD = Food, DR = Drinks, NC = Non-Consumable)
Item_Weight | Weight of the product
Item_Fat_Content | Whether the product is low fat or not
Item_Visibility | % of the store's display area given to this product
Item_Type | Category of the product
Item_MRP | Maximum Retail Price (list price)
Outlet_Identifier | Unique store ID
Outlet_Establishment_Year | Year the store was opened
Outlet_Size | Size of the store (Small / Medium / High)
Outlet_Location_Type | Type of city (Tier 1 / 2 / 3)
Outlet_Type | Grocery store or Supermarket Type 1 / 2 / 3
Item_Outlet_Sales | **Target**: sales of the product in that store

## Import modules

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
%matplotlib inline

## Loading the dataset
`train` has the target (sales). `test` does not, so we predict it at the end.

In [ ]:
TRAIN_PATH = 'train_v9rqX0R.csv'   # change if your file name is different
TEST_PATH  = 'test_AbJTz2l.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print('Train shape:', train.shape, '| Test shape:', test.shape)
train.head()

In [ ]:
# statistical info
train.describe()

In [ ]:
# datatype of attributes
train.info()

In [ ]:
# number of unique values in each column
train.apply(lambda x: len(x.unique()))

## Preprocessing the dataset
We clean **train and test together** so both get exactly the same treatment. `is_train` lets us separate them again later.

In [ ]:
train['is_train'] = 1
test['is_train'] = 0
data = pd.concat([train, test], ignore_index=True)
data.shape

**Step 1: Check for null values**

In [ ]:
data.drop(columns='Item_Outlet_Sales').isnull().sum()

Two columns have missing values: `Item_Weight` and `Outlet_Size`. (Sales is empty only for test rows, which is expected.)

**Step 2: Fix the messy labels in `Item_Fat_Content`**

In [ ]:
print(data['Item_Fat_Content'].value_counts())
data['Item_Fat_Content'] = data['Item_Fat_Content'].replace({'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})
print()
print(data['Item_Fat_Content'].value_counts())

**Step 3: Fill missing `Item_Weight`**
The same product always has the same weight, so we copy it from the product's other rows. Anything still missing gets the overall mean.

In [ ]:
data['Item_Weight'] = data['Item_Weight'].fillna(data.groupby('Item_Identifier')['Item_Weight'].transform('mean'))
data['Item_Weight'] = data['Item_Weight'].fillna(data['Item_Weight'].mean())
data['Item_Weight'].isnull().sum()

**Step 4: Fill missing `Outlet_Size`**
We fill with the most common size of the same `Outlet_Type`.

In [ ]:
size_mode = data.groupby('Outlet_Type')['Outlet_Size'].agg(lambda x: x.mode()[0])
print(size_mode)
miss = data['Outlet_Size'].isnull()
data.loc[miss, 'Outlet_Size'] = data.loc[miss, 'Outlet_Type'].map(size_mode)
data['Outlet_Size'].isnull().sum()

**Step 5: `Item_Visibility` = 0 is impossible** (a product on a shelf has some visibility). We treat 0 as missing and fill it with that product's average visibility.

In [ ]:
print('Zeros before:', (data['Item_Visibility'] == 0).sum())
data['Item_Visibility'] = data['Item_Visibility'].replace(0, np.nan)
data['Item_Visibility'] = data['Item_Visibility'].fillna(data.groupby('Item_Identifier')['Item_Visibility'].transform('mean'))
data['Item_Visibility'] = data['Item_Visibility'].fillna(data['Item_Visibility'].mean())
print('Zeros after :', (data['Item_Visibility'] == 0).sum())

## Creation of New Attributes (Feature Engineering)
Feature engineering = building new, more useful columns from the existing ones.

In [ ]:
# 1. Broad category from the ID prefix: FD / DR / NC
data['Item_Category'] = data['Item_Identifier'].str[:2].map({'FD': 'Food', 'DR': 'Drinks', 'NC': 'Non-Consumable'})
print(data['Item_Category'].value_counts())

# Fat content has no meaning for non-consumables
data.loc[data['Item_Category'] == 'Non-Consumable', 'Item_Fat_Content'] = 'Non-Edible'

In [ ]:
# 2. Age of the outlet (data is from 2013)
data['Outlet_Years'] = 2013 - data['Outlet_Establishment_Year']

# 3. Visibility relative to that product's own average (is it displayed more than usual?)
data['Visibility_Ratio'] = data['Item_Visibility'] / data.groupby('Item_Identifier')['Item_Visibility'].transform('mean')

# 4. Price per unit of weight
data['MRP_per_Weight'] = data['Item_MRP'] / data['Item_Weight']

# 5. log of price. Sales = price x quantity, so on a log scale it becomes price + quantity
data['log_MRP'] = np.log(data['Item_MRP'])

# 6. Price band
data['MRP_Band'] = pd.cut(data['Item_MRP'], [0, 69, 136, 203, 400], labels=['Low', 'Medium', 'High', 'Premium']).astype(str)

data[['Item_Identifier', 'Item_Category', 'Outlet_Years', 'Visibility_Ratio', 'MRP_per_Weight', 'log_MRP', 'MRP_Band']].head()

## Exploratory Data Analysis
We use only the training rows here (they have the target).

In [ ]:
tr = data[data['is_train'] == 1].copy()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(tr['Item_Weight'], kde=True, ax=ax[0]);      ax[0].set_title('Item_Weight')
sns.histplot(tr['Item_Visibility'], kde=True, ax=ax[1]);  ax[1].set_title('Item_Visibility (right-skewed)')
sns.histplot(tr['Item_MRP'], kde=True, ax=ax[2]);         ax[2].set_title('Item_MRP (4 clusters = price bands)')
plt.tight_layout(); plt.show()

**Target variable.** Sales is heavily right-skewed (many small values, few very large). Linear models work better on a bell-shaped target, so we take a **log transform**: `log(1 + sales)`.

In [ ]:
print('Skewness before log:', round(tr['Item_Outlet_Sales'].skew(), 2))
print('Skewness after  log:', round(np.log1p(tr['Item_Outlet_Sales']).skew(), 2))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(tr['Item_Outlet_Sales'], kde=True, ax=ax[0]);            ax[0].set_title('Sales (skewed)')
sns.histplot(np.log1p(tr['Item_Outlet_Sales']), kde=True, ax=ax[1]);  ax[1].set_title('log(1 + Sales) (closer to normal)')
plt.tight_layout(); plt.show()

**Categorical columns**

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 8))
for a, col in zip(ax.ravel(), ['Item_Fat_Content', 'Item_Category', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'MRP_Band']):
    sns.countplot(data=tr, x=col, ax=a); a.set_title(col); a.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
sns.countplot(data=tr, x='Item_Type', order=tr['Item_Type'].value_counts().index)
plt.xticks(rotation=90); plt.title('Item_Type'); plt.show()

**Relationship with the target** (this is the most important EDA)
- Price (`Item_MRP`) vs sales
- Sales by store type and by store

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(tr['Item_MRP'], tr['Item_Outlet_Sales'], s=4, alpha=0.4)
ax[0].set_title('Item_MRP vs Sales: sales grow with price'); ax[0].set_xlabel('Item_MRP')
sns.boxplot(data=tr, x='Outlet_Type', y='Item_Outlet_Sales', ax=ax[1])
ax[1].set_title('Sales by Outlet_Type'); ax[1].tick_params(axis='x', rotation=20)
sns.boxplot(data=tr, x='Outlet_Identifier', y='Item_Outlet_Sales', ax=ax[2])
ax[2].set_title('Sales by Outlet'); ax[2].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()

**What the plots tell us**
- Sales rise with price, in separate bands.
- The store matters a lot: grocery stores sell far less than supermarkets.
- So a good model must capture **price** and **store**.

## Correlation Matrix

In [ ]:
num_cols = ['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Establishment_Year', 'Outlet_Years',
            'Visibility_Ratio', 'MRP_per_Weight', 'log_MRP', 'Item_Outlet_Sales']
plt.figure(figsize=(9, 7))
sns.heatmap(tr[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation matrix'); plt.show()

Reading the heatmap:
- `Item_MRP` has the strongest correlation with sales.
- `Item_MRP` and `log_MRP` are almost the same column, and `Outlet_Establishment_Year` and `Outlet_Years` are perfectly (negatively) correlated. This is **multicollinearity**, which we handle next.

## Multicollinearity
**Multicollinearity** = two or more input features carry (almost) the same information. In linear regression it makes the coefficients unstable and hard to interpret (a small change in the data can flip a coefficient's sign).

**How we detect it:** the **Variance Inflation Factor (VIF)**. For each feature, predict it from all the other features. If it can be predicted well (high R^2), VIF = 1 / (1 - R^2) is large.

| VIF | Meaning |
|---|---|
| 1 | No collinearity |
| 1 - 5 | Fine |
| 5 - 10 | Moderate, watch it |
| > 10 | Serious, remove the feature |

**(a) Categorical redundancy:** are the outlet columns just copies of `Outlet_Identifier`?

In [ ]:
for c in ['Outlet_Type', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Establishment_Year']:
    fixed = (tr.groupby('Outlet_Identifier')[c].nunique() == 1).all()
    print(f'{c:28s} fully decided by Outlet_Identifier: {fixed}')

Yes. Every store has exactly one type, size, location and opening year, so `Outlet_Identifier` already contains all of that information. We will use `Outlet_Identifier` only and drop the others.

**(b) Numeric features: VIF**

In [ ]:
def compute_vif(X):
    vifs = {}
    for col in X.columns:
        others = X.drop(columns=col)
        r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
        vifs[col] = np.inf if r2 > 0.999999 else 1 / (1 - r2)
    return pd.Series(vifs).sort_values(ascending=False)

numeric_features = ['Item_Weight', 'Item_Visibility', 'Item_MRP', 'log_MRP', 'Outlet_Years',
                    'Outlet_Establishment_Year', 'Visibility_Ratio', 'MRP_per_Weight']
print('VIF before removing anything:')
compute_vif(tr[numeric_features]).round(2)

Now we remove the feature with the highest VIF, recompute, and repeat until every VIF is below 10.

In [ ]:
selected = list(numeric_features)
while True:
    vif = compute_vif(tr[selected])
    if vif.iloc[0] <= 10:
        break
    print(f'Dropping {vif.index[0]:28s} (VIF = {vif.iloc[0]:.1f})')
    selected.remove(vif.index[0])

print('\nFinal numeric features:', selected)
compute_vif(tr[selected]).round(2)

Result: duplicates such as `Outlet_Establishment_Year` (a perfect copy of `Outlet_Years`) and other overlapping price/visibility columns are removed. The remaining features are largely independent, so the linear models become stable.

*Note: Random Forest is not hurt by multicollinearity, but we use the same cleaned feature set for both models so the comparison is fair.*

## Onehot Encoding
Models need numbers, so each category becomes 0/1 columns. We use `drop_first=True` to remove one column per category. Without it, the dummy columns always add up to 1 and are perfectly collinear (the **dummy variable trap**).

In [ ]:
cat_features = ['Outlet_Identifier', 'Item_Category', 'Item_Fat_Content', 'MRP_Band']
X_all = pd.get_dummies(data[selected + cat_features], columns=cat_features, drop_first=True).astype(float)
print(X_all.shape)
X_all.head()

## Input Split
Separate the training rows (with target) from the test rows again. The target is `log(1 + sales)`.

In [ ]:
X = X_all[data['is_train'] == 1].reset_index(drop=True)
X_test = X_all[data['is_train'] == 0].reset_index(drop=True)

y_raw = train['Item_Outlet_Sales'].reset_index(drop=True)
y = np.log1p(y_raw)
X.shape, X_test.shape

## Model Training
For every model we report, on the **original rupee scale** (so it is comparable with the competition):
- **Train RMSE**: error on the data the model has seen
- **CV RMSE**: error on unseen data (5-fold cross-validation). **This is the number that matters.**
- **CV R^2**: share of the variation in sales explained

A big gap between Train RMSE and CV RMSE means **overfitting**.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

def train_model(name, model):
    model.fit(X, y)
    train_pred = np.expm1(model.predict(X))
    cv_pred = np.expm1(cross_val_predict(model, X, y, cv=kf))        # predictions on unseen folds
    rmse = lambda p: float(np.sqrt(np.mean((y_raw - p) ** 2)))
    cv_r2 = 1 - np.sum((y_raw - cv_pred) ** 2) / np.sum((y_raw - y_raw.mean()) ** 2)
    results[name] = {'Train RMSE': rmse(train_pred), 'CV RMSE': rmse(cv_pred), 'CV R2': cv_r2}
    print(f'{name}\n  Train RMSE: {rmse(train_pred):.2f}\n  CV RMSE   : {rmse(cv_pred):.2f}\n  CV R2     : {cv_r2:.3f}')

### 1. Linear Regression
The simplest model: sales is a weighted sum of the inputs. Each coefficient tells how much sales change when that input increases, so it is easy to explain.

In [ ]:
model = LinearRegression()
train_model('Linear Regression', model)
coef = pd.Series(model.coef_, X.columns).sort_values()
coef.plot(kind='bar', figsize=(12, 4), title='Linear Regression coefficients'); plt.show()

### 2. Random Forest
Many decision trees on random samples, predictions averaged. `min_samples_leaf=5` stops the trees from memorising the data.

In [ ]:
model = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42)
train_model('Random Forest', model)
pd.Series(model.feature_importances_, X.columns).sort_values(ascending=False).head(12).plot(
    kind='bar', figsize=(12, 4), title='Random Forest feature importance'); plt.show()

### Model comparison

In [ ]:
comparison = pd.DataFrame(results).T.sort_values('CV RMSE')
display(comparison.round(3))
comparison['CV RMSE'].plot(kind='barh', figsize=(8, 3.5), title='CV RMSE (lower is better)'); plt.show()
best_name = comparison.index[0]
print('Best model by CV RMSE:', best_name)

## Predictions on the test set
We refit the best model and convert predictions back from log scale with `expm1` (the reverse of `log1p`).

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42),
}
final_model = models[best_name].fit(X, y)
pred_baseline = np.clip(np.expm1(final_model.predict(X_test)), 0, None)

submission_baseline = test[['Item_Identifier', 'Outlet_Identifier']].copy()
submission_baseline['Item_Outlet_Sales'] = pred_baseline
submission_baseline.to_csv('submission_baseline.csv', index=False)
submission_baseline.head()

## Our Final Model: Structural (Price x Store x Popularity) Model
The models above are standard machine learning. Our team's final model uses a structure we found in the data.

**Idea in plain words**
1. A sale is `units sold x unit price`.
2. Every sale is a whole number of units, so every sale is a **multiple of one small currency step**. That step is the greatest common divisor (GCD) of all training sales, about 0.6658.
3. For each product, exactly one price close to its MRP fits that step. That is the product's **unit price**.
4. Dividing sales by unit price gives **units sold**, a much cleaner number to model.
5. Prediction = `unit price x store rate x product popularity`
   - **store rate** = average units sold per row in that store
   - **product popularity** = how much better than its stores' average the product sells, **shrunk** towards 1 for products with few rows so a lucky product doesn't get an extreme score

Only the training file is used to learn all of this.

In [ ]:
SMOOTHING = 55.234948951954145   # tuned by cross-validation
OFFSETS = np.arange(-20, 21) / 10  # candidate unit price = MRP + offset

def currency_step(sales):
    return float(np.gcd.reduce(np.rint(np.asarray(sales, float) * 10000).astype(np.int64))) / 10000

def unit_price(mrp, step):
    mrp = np.asarray(mrp, float)
    if not 0.1 <= step <= 5.0:
        return mrp
    candidates = mrp[:, None] + OFFSETS[None, :]
    on_step = np.isclose(candidates / step, np.round(candidates / step), rtol=0, atol=1e-7)
    unique = on_step.sum(axis=1) == 1
    return np.where(unique, candidates[np.arange(len(mrp)), on_step.argmax(axis=1)], mrp)

def fit_structural(tr_df):
    step = currency_step(tr_df['Item_Outlet_Sales'])
    units = tr_df['Item_Outlet_Sales'].to_numpy(float) / unit_price(tr_df['Item_MRP'], step)
    stores = tr_df['Outlet_Identifier'].to_numpy()
    rates = pd.Series(units).groupby(stores).mean()                         # store rate
    relative = units / pd.Series(stores).map(rates).to_numpy()
    stats = pd.Series(relative).groupby(tr_df['Item_Identifier'].to_numpy()).agg(['sum', 'count'])
    popularity = (stats['sum'] + SMOOTHING) / (stats['count'] + SMOOTHING)  # shrunk popularity
    return step, rates, popularity, float(units.mean())

def predict_structural(model, rows):
    step, rates, popularity, default_rate = model
    price = unit_price(rows['Item_MRP'], step)
    rate = rows['Outlet_Identifier'].map(rates).fillna(default_rate).to_numpy(float)
    prod = rows['Item_Identifier'].map(popularity).fillna(1.0).to_numpy(float)
    return np.clip(price * rate * prod, 0, None)

**Learned currency step** (the GCD of all training sales):

In [ ]:
structural = fit_structural(train)
print('Currency step learned from training sales:', structural[0])
print('\nStore rates (average units per row):')
structural[1].round(2)

**Cross-validated RMSE** with the same 5 folds, so it is directly comparable with the baseline models above:

In [ ]:
oof = np.zeros(len(train))
for tr_idx, va_idx in kf.split(train):
    m = fit_structural(train.iloc[tr_idx])
    oof[va_idx] = predict_structural(m, train.iloc[va_idx])

struct_rmse = float(np.sqrt(np.mean((y_raw - oof) ** 2)))
print(f'Structural model CV RMSE: {struct_rmse:.2f}')
print(f'Best baseline ({best_name}) CV RMSE: {comparison.loc[best_name, "CV RMSE"]:.2f}')

In [ ]:
final_comparison = comparison[['CV RMSE']].copy()
final_comparison.loc['Structural (our final model)'] = struct_rmse
final_comparison = final_comparison.sort_values('CV RMSE')
final_comparison['CV RMSE'].plot(kind='barh', figsize=(9, 4), title='All models: CV RMSE (lower is better)'); plt.show()
final_comparison.round(2)

In [ ]:
pred_struct = predict_structural(structural, test)
submission_final = test[['Item_Identifier', 'Outlet_Identifier']].copy()
submission_final['Item_Outlet_Sales'] = pred_struct
submission_final.to_csv('submission_final_structural.csv', index=False)
submission_final.head()

## Conclusion
- **EDA** showed that sales depend mainly on **price** and **store**.
- **Cleaning** fixed inconsistent labels, missing weights/sizes and impossible zero visibilities.
- **Feature engineering** added item category, outlet age, visibility ratio, price per weight, log price and price bands.
- **Multicollinearity** (VIF and redundant outlet columns) was removed so linear models are stable.
- **Two baseline models** (Linear Regression and Random Forest) were compared with 5-fold cross-validation.
- **Our final structural model** uses `sales = units x price`, learns store rates and a shrunk product popularity, and is simple to explain.